# Apache Flink for Data Engineers
## True Stream Processing, Stateful Computation & Event-Time Semantics

**What you'll learn:**
- Flink architecture: JobManager, TaskManagers, parallelism
- True streaming vs micro-batch (Flink vs Spark paradigm)
- Windowing: tumbling, sliding, session windows
- Event time, watermarks, and late data handling
- State management: keyed state, state backends, TTL
- Checkpointing and exactly-once fault tolerance
- Flink SQL and Table API
- Flink + Kafka integration patterns
- Flink on AWS (Amazon Managed Service for Apache Flink)
- When to choose Flink vs Spark
- Interview questions

**Estimated Time:** 6-8 hours

---

> Flink processes events ONE AT A TIME as they arrive (true streaming).
> Spark processes events in MICRO-BATCHES (collects, then processes).
> This fundamental difference determines when you choose each.

---

---
# Section 1: Flink Architecture

## Core Components

```
CLIENT (submits job)
    |
    v
JOB MANAGER (master)
├── Dispatcher: receives jobs, launches JobMasters
├── ResourceManager: manages TaskManager slots
└── JobMaster: manages execution of a single job
    |
    v
TASK MANAGERS (workers, 1 per machine)
├── Slots: fixed number of parallel execution units
│   ├── Slot 1: [Operator Chain: Source → Map → Filter]
│   ├── Slot 2: [Operator Chain: Source → Map → Filter]
│   └── Slot 3: [KeyedProcess → Sink]
└── Memory: Managed (off-heap) + JVM heap

CHECKPOINTING (periodic snapshots)
├── State Backend: RocksDB (disk) or HashMapStateBackend (memory)
└── Checkpoint Storage: S3, HDFS, or local filesystem
```

## Key Concepts

| Concept | Description |
|---------|-------------|
| **JobGraph** | Logical plan of operators and data flow |
| **ExecutionGraph** | Physical plan (parallelized) |
| **Task Slot** | Unit of parallelism on a TaskManager |
| **Operator Chain** | Fused operators (no network shuffle) |
| **Parallelism** | How many parallel instances of an operator |
| **Checkpoint** | Consistent snapshot of all operator states |
| **Savepoint** | Manual checkpoint (for upgrades, migrations) |

## Execution Model: Continuous Processing

```
SPARK (micro-batch):
  Collect 5 seconds of data → Process batch → Output → Repeat
  Latency: seconds to minutes

FLINK (continuous):
  Event arrives → Process immediately → Output immediately
  Latency: milliseconds

SPARK (Structured Streaming with Continuous mode):
  Approaching Flink's latency but less mature
```

---
# Section 2: Flink vs Spark — Detailed Comparison

| Feature | Apache Flink | Apache Spark |
|---------|-------------|--------------|
| **Processing model** | True streaming (event-at-a-time) | Micro-batch (Structured Streaming) |
| **Latency** | Milliseconds | Seconds to minutes |
| **State management** | First-class (keyed state, TTL, queryable) | Limited (mapGroupsWithState) |
| **Windowing** | Rich (tumbling, sliding, session, global) | Good (tumbling, sliding, session) |
| **Event time** | Native, robust watermark handling | Supported but less mature |
| **Exactly-once** | Native (Chandy-Lamport checkpointing) | Via checkpointing + idempotent sinks |
| **SQL** | Flink SQL (streaming + batch) | Spark SQL (primarily batch, streaming limited) |
| **Batch processing** | Supported (bounded streams) | Excellent (primary strength) |
| **Ecosystem** | Smaller, streaming-focused | Huge (ML, Graph, SQL, streaming) |
| **Deployment** | Standalone, YARN, K8s, AWS Managed | Standalone, YARN, K8s, Databricks |
| **Language APIs** | Java, Scala, Python (PyFlink) | Python, Scala, Java, R, SQL |
| **Community** | Growing fast | Very large and mature |
| **Managed services** | AWS Managed Flink, Ververica | Databricks, EMR, Dataproc, Synapse |

## When to Choose Flink

- Sub-second latency requirements (fraud detection, alerting)
- Complex event processing (patterns across events)
- Large stateful computations (sessionization at scale)
- True event-time processing with complex watermarks
- You need queryable state (serve state to external apps)

## When to Choose Spark

- Batch + streaming in same codebase (Lakehouse pattern)
- Team already knows Spark/PySpark
- Heavy batch ETL with occasional streaming
- ML pipelines integrated with data processing
- Databricks ecosystem (Delta Lake, Unity Catalog, MLflow)

## The Reality in 2026

Most companies use BOTH:
- **Spark** for batch ETL, data lake management, ML
- **Flink** for real-time (fraud, recommendations, alerting)
- **Kafka** connects them both

In [ ]:
# Flink programming model (conceptual Python/PyFlink)
print("FLINK PROGRAMMING MODEL (PyFlink Concepts)")
print("=" * 60)

print("""
# PyFlink DataStream API (conceptual -- requires Flink runtime)

from pyflink.datastream import StreamExecutionEnvironment
from pyflink.common import WatermarkStrategy, Time

# 1. Create execution environment
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(4)
env.enable_checkpointing(60000)  # Checkpoint every 60 seconds

# 2. Define source (Kafka)
kafka_source = KafkaSource.builder() \
    .set_bootstrap_servers("broker:9092") \
    .set_topics("orders") \
    .set_group_id("flink_etl") \
    .set_starting_offsets(KafkaOffsetsInitializer.earliest()) \
    .set_value_only_deserializer(JsonRowDeserializationSchema(...)) \
    .build()

# 3. Read stream with watermarks (event time)
orders_stream = env.from_source(
    kafka_source,
    WatermarkStrategy.for_bounded_out_of_orderness(Duration.of_seconds(5)),
    "Kafka Orders Source"
)

# 4. Transformations
processed = (orders_stream
    .filter(lambda order: order['status'] == 'completed')
    .key_by(lambda order: order['customer_id'])  # Partition by key
    .window(TumblingEventTimeWindows.of(Time.minutes(5)))  # 5-min windows
    .reduce(lambda a, b: {
        'customer_id': a['customer_id'],
        'total': a['total'] + b['total'],
        'count': a['count'] + 1
    })
)

# 5. Sink (write results)
processed.add_sink(kafka_sink)  # Or JDBC sink, Delta sink, etc.

# 6. Execute (nothing happens until this!)
env.execute("Order Processing Pipeline")
""")

print("KEY DIFFERENCES FROM SPARK:")
print("  - env.execute() triggers the CONTINUOUS pipeline")
print("  - filter/map/keyBy/window run CONTINUOUSLY (not in batches)")
print("  - .key_by() = Spark's .groupBy() but for streaming state")
print("  - Watermarks tell Flink 'how late data can be'")

---
# Section 4: Windowing in Flink

## Window Types

```
TUMBLING WINDOW (fixed, non-overlapping):
  |--5min--|--5min--|--5min--|--5min--|
  |  W1    |  W2    |  W3    |  W4    |
  Events fall into exactly ONE window.

SLIDING WINDOW (fixed, overlapping):
  |----10min----|
       |----10min----|
            |----10min----|
  Slide every 5 minutes. Events can be in MULTIPLE windows.

SESSION WINDOW (dynamic, gap-based):
  |--session 1--|   gap   |--session 2--|   gap   |--sess 3--|
  Events within 30min of each other = same session.
  Each key has INDEPENDENT sessions.

GLOBAL WINDOW (custom trigger):
  |-------------------- all events -------------------|
  YOU define when to fire (every N events, time-based, custom).
```

## Event Time vs Processing Time

```
Event Time:   When the event ACTUALLY happened (embedded timestamp)
              "This order was placed at 14:30:05"
              Used for: Correct analytics regardless of arrival delay

Processing Time: When Flink RECEIVES the event
                 "Flink saw this event at 14:30:08" (3s late)
                 Used for: Low-latency, order doesn't matter

Ingestion Time: When event enters Flink (rarely used)
```

## Watermarks: Handling Late Data

```
Watermark = "I believe all events up to time T have arrived"

Stream: e(14:30) e(14:31) e(14:32) [watermark: 14:30] e(14:33) ...
                                          ^
                                    "All events up to 14:30 are here"
                                    Triggers window [14:25-14:30]

Late event arrives at 14:35 with event_time=14:29:
  - If watermark already past 14:30 → event is LATE
  - Options: drop it, send to side output, update result

Bounded out-of-orderness watermark:
  watermark = max_event_time_seen - allowed_lateness
  Example: 5 seconds allowed → events up to 5s late are included
```

In [ ]:
# Windowing simulation
print("FLINK WINDOWING -- Simulated")
print("=" * 60)

from collections import defaultdict

# Simulate tumbling window aggregation
events = [
    {"customer": "A", "amount": 100, "event_time": 1.0},
    {"customer": "B", "amount": 200, "event_time": 2.0},
    {"customer": "A", "amount": 150, "event_time": 3.5},
    {"customer": "A", "amount": 80, "event_time": 5.5},  # Next window
    {"customer": "B", "amount": 300, "event_time": 6.0},
    {"customer": "A", "amount": 120, "event_time": 8.0},
    {"customer": "B", "amount": 50, "event_time": 9.5},
    {"customer": "A", "amount": 200, "event_time": 11.0},  # Next window
]

window_size = 5.0  # 5-second tumbling windows

# Assign events to windows
windows = defaultdict(lambda: defaultdict(list))
for event in events:
    window_start = int(event["event_time"] // window_size) * window_size
    window_key = f"[{window_start:.0f}-{window_start + window_size:.0f})"
    windows[window_key][event["customer"]].append(event["amount"])

print("  Tumbling windows (5-second):")
for window, customers in sorted(windows.items()):
    print(f"\n  Window {window}:")
    for customer, amounts in sorted(customers.items()):
        print(f"    Customer {customer}: {amounts} -> total={sum(amounts)}")

# Session window simulation
print("\n\n  Session windows (gap > 3 seconds = new session):")
gap_threshold = 3.0
customer_a_events = sorted(
    [e for e in events if e["customer"] == "A"],
    key=lambda x: x["event_time"]
)

sessions = []
current_session = [customer_a_events[0]]
for event in customer_a_events[1:]:
    if event["event_time"] - current_session[-1]["event_time"] > gap_threshold:
        sessions.append(current_session)
        current_session = [event]
    else:
        current_session.append(event)
sessions.append(current_session)

for i, session in enumerate(sessions):
    times = [e["event_time"] for e in session]
    amounts = [e["amount"] for e in session]
    print(f"    Session {i+1}: times={times}, amounts={amounts}, total={sum(amounts)}")

---
# Section 5: State Management & Checkpointing

## Keyed State (per-key storage)

```python
# In Flink, each key (e.g., customer_id) has its own state
class OrderCounter(KeyedProcessFunction):
    def open(self, ctx):
        # State is automatically scoped to the current key
        self.count = ctx.get_state(ValueStateDescriptor("count", Types.LONG()))
        self.total = ctx.get_state(ValueStateDescriptor("total", Types.DOUBLE()))

    def process_element(self, order, ctx):
        # This runs for ONE key (one customer)
        current_count = self.count.value() or 0
        current_total = self.total.value() or 0.0

        self.count.update(current_count + 1)
        self.total.update(current_total + order.amount)

        # State is checkpointed automatically!
```

## State Backends

| Backend | Storage | Speed | Use When |
|---------|---------|-------|----------|
| **HashMapStateBackend** | JVM heap memory | Fastest | Small state (<10GB) |
| **RocksDB** | Disk (SSD) + memory cache | Fast | Large state (TBs), incremental checkpoints |

## Checkpointing: Exactly-Once Semantics

```
Flink's Checkpointing (simplified Chandy-Lamport):

1. JobManager sends CHECKPOINT BARRIER to all sources
2. Barriers flow through the DAG like regular events
3. When operator receives barrier from ALL inputs:
   - Snapshot its state to durable storage (S3, HDFS)
   - Forward barrier to downstream operators
4. When all operators complete: checkpoint is DONE

On FAILURE:
   - All operators restore from last completed checkpoint
   - Sources replay from checkpointed offsets (Kafka: seek to offset)
   - Processing resumes with NO data loss or duplication

Key insight: This is DIFFERENT from Spark's micro-batch recovery.
Flink recovers MID-STREAM, Spark re-runs the entire micro-batch.
```

## Savepoints vs Checkpoints

| | Checkpoint | Savepoint |
|--|-----------|-----------|
| **Triggered by** | Automatic (periodic) | Manual (operator/API) |
| **Purpose** | Fault tolerance | Upgrades, migrations, A/B testing |
| **Lifecycle** | Auto-deleted when superseded | Persists until manually deleted |
| **Use case** | Crash recovery | Deploy new version of job |

In [ ]:
# Flink SQL concepts
print("FLINK SQL -- Streaming SQL")
print("=" * 60)

print("""
-- Flink SQL treats streams as DYNAMIC TABLES:
-- A stream is an ever-growing table (new rows appear continuously)

-- CREATE SOURCE TABLE (reads from Kafka)
CREATE TABLE orders_source (
    order_id STRING,
    customer_id STRING,
    amount DECIMAL(10,2),
    order_time TIMESTAMP(3),
    -- Watermark: allows 5 seconds of lateness
    WATERMARK FOR order_time AS order_time - INTERVAL '5' SECOND
) WITH (
    'connector' = 'kafka',
    'topic' = 'orders',
    'properties.bootstrap.servers' = 'broker:9092',
    'format' = 'json',
    'scan.startup.mode' = 'earliest-offset'
);

-- TUMBLING WINDOW AGGREGATION (continuous!)
SELECT
    customer_id,
    TUMBLE_START(order_time, INTERVAL '5' MINUTE) AS window_start,
    TUMBLE_END(order_time, INTERVAL '5' MINUTE) AS window_end,
    COUNT(*) AS order_count,
    SUM(amount) AS total_amount
FROM orders_source
GROUP BY
    customer_id,
    TUMBLE(order_time, INTERVAL '5' MINUTE);

-- This query runs CONTINUOUSLY, emitting results as windows close!

-- PATTERN MATCHING (MATCH_RECOGNIZE -- unique to Flink SQL!)
-- Detect fraud: 3 failed payments followed by a success
SELECT *
FROM payments
MATCH_RECOGNIZE (
    PARTITION BY customer_id
    ORDER BY event_time
    MEASURES
        FIRST(A.amount) AS first_fail_amount,
        LAST(A.amount) AS last_fail_amount,
        B.amount AS success_amount
    PATTERN (A{3,} B)
    DEFINE
        A AS A.status = 'failed',
        B AS B.status = 'success'
);
""")

print("KEY FLINK SQL FEATURES:")
print("  - WATERMARK: built into table definition")
print("  - TUMBLE/HOP/SESSION: window functions for streams")
print("  - MATCH_RECOGNIZE: CEP (Complex Event Processing) in SQL")
print("  - Temporal Joins: join stream with versioned table")
print("  - These run CONTINUOUSLY (not one-shot like Spark SQL)")

---
# Section 7: Flink on AWS (Amazon Managed Service for Apache Flink)

## Deployment Options

| Option | Description | Use When |
|--------|-------------|----------|
| **Amazon Managed Flink** | Fully managed, serverless | Production streaming, no cluster management |
| **EMR (Flink on YARN)** | Self-managed on EMR | Need full control, mixed Spark+Flink |
| **EKS (Flink on K8s)** | Containerized, self-managed | Cloud-native, multi-cloud |
| **Self-hosted EC2** | Full DIY | Development, testing |

## Amazon Managed Flink (formerly Kinesis Data Analytics)

```
Architecture:
  Source (Kafka/Kinesis/S3) → Managed Flink → Sink (S3/Redshift/DynamoDB/Kafka)

Key Features:
  - Auto-scaling (KPU = Kinesis Processing Units)
  - Automatic checkpointing to S3
  - VPC integration for private sources/sinks
  - CloudWatch metrics built-in
  - Supports Java, Scala, Python (PyFlink), and SQL

Pricing: Pay per KPU-hour ($0.11/hr per KPU)
  1 KPU = 1 vCPU + 4GB memory
  Typical small job: 2-4 KPUs = $0.22-0.44/hr
```

## Common AWS Streaming Architecture

```
Producers                     Streaming                   Storage/Analytics
┌──────────┐               ┌──────────────┐            ┌──────────────┐
│ App/API  │──Kafka/MSK──>│ Managed Flink │──────────>│ S3 (Delta)   │
│ CDC      │               │ (processing) │            │ Redshift     │
│ IoT Core │──Kinesis────>│              │──────────>│ DynamoDB     │
│ CloudWatch│              └──────────────┘            │ OpenSearch   │
└──────────┘                     │                     └──────────────┘
                                 │ Alerts
                                 ▼
                          ┌──────────────┐
                          │ SNS/Lambda   │
                          │ (alerting)   │
                          └──────────────┘
```

In [ ]:
print("FLINK INTERVIEW QUESTIONS")
print("=" * 60)

questions = [
    {
        "q": "How does Flink achieve exactly-once processing?",
        "a": "Flink uses the Chandy-Lamport algorithm for distributed snapshots. Checkpoint barriers flow through the stream. When all operators have snapshotted their state, the checkpoint is complete. On failure, all operators restore from the last checkpoint and sources replay from stored offsets. Combined with idempotent/transactional sinks, this gives end-to-end exactly-once."
    },
    {
        "q": "What are watermarks and why are they needed?",
        "a": "Watermarks track event-time progress in a stream. A watermark W(t) means: all events with timestamp <= t have arrived. This tells window operators when to fire. Without watermarks, windows would never close (always waiting for more data). Watermarks handle out-of-order events by allowing bounded lateness."
    },
    {
        "q": "Flink vs Spark Structured Streaming: when to use each?",
        "a": "Flink: sub-second latency, complex stateful processing, true event-time, CEP patterns. Spark: batch+streaming unified, Lakehouse integration (Delta Lake), team already uses Spark, ML pipelines. In practice: Flink for real-time critical paths (fraud, alerting), Spark for ETL and batch-first pipelines."
    },
    {
        "q": "What is backpressure in Flink and how is it handled?",
        "a": "Backpressure occurs when a downstream operator is slower than upstream. Flink handles it via credit-based flow control: downstream operators tell upstream how many buffers they can accept. This naturally slows producers without data loss. Monitor via Flink UI backpressure metrics."
    },
    {
        "q": "Explain the difference between keyed state and operator state.",
        "a": "Keyed state is partitioned by key (each key has isolated state). Used for per-entity aggregations, sessions, etc. Operator state is per-operator-instance (not key-partitioned). Used for source offsets, broadcast state, or list-based state that doesn't partition by key."
    },
    {
        "q": "How would you upgrade a Flink job without losing state?",
        "a": "Take a SAVEPOINT (manual checkpoint), stop the old job, deploy new version pointing to the savepoint. The new job restores state from savepoint and resumes. Requirements: operator UIDs must be stable, state schema must be compatible (or use state migration)."
    },
]

for i, qa in enumerate(questions, 1):
    print(f"\nQ{i}: {qa['q']}")
    print(f"{'─'*60}")
    print(f"  {qa['a']}")

---
# Summary: Flink Cheat Sheet

## When to Use Flink

| Scenario | Flink? | Why |
|----------|--------|-----|
| Sub-second alerting | Yes | True streaming, ms latency |
| Fraud detection | Yes | Stateful patterns (MATCH_RECOGNIZE) |
| Real-time sessionization | Yes | Native session windows + state |
| Batch ETL to Delta Lake | No | Use Spark |
| ML training pipeline | No | Use Spark MLlib |
| Mixed batch + streaming | Depends | Spark simpler, Flink more capable for streaming |
| CDC processing | Both | Flink for real-time, Spark for micro-batch |

## Key Flink Concepts for Interviews

- **True streaming**: events processed one-at-a-time (not batched)
- **Checkpointing**: distributed snapshots for exactly-once
- **Watermarks**: event-time progress tracking
- **Keyed State**: per-entity state (like a distributed hash map)
- **Savepoints**: manual checkpoints for job upgrades
- **Operator Chaining**: fuse operators to avoid network overhead
- **Backpressure**: natural flow control via credits

---
*Apache Flink for Data Engineers*